# Earnalism stills job `stills-film-01-r7-candidates2`

**Prepared, not run.** Input zip SHA256 `1abd56dab70e580c1bfb28d612d006d937a01b6efe935b7927ce2a321a7aa8cb`. Copy `inputs.zip` to `/content/drive/MyDrive/earnalism-audiovisual/bicharak/stills-film-01-r7-candidates2/` in Drive, set `RUN = True` in the first cell, then run all. Outputs and `manifest.json` (sha256 of every output) land in the same folder.

**Hard stop:** 1780 s (the reserved forecast) from the clock cell. No item starts after it and the runtime is unassigned at it; a stopped run's manifest says INCOMPLETE and cannot be imported.

**Disconnected?** Run all again: finished items are kept (`progress.jsonl`, checked by sha256) and the run continues where it stopped, within the same reserved time (`sessions.jsonl` counts what earlier runs used).


In [ ]:
RUN = False

In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import pathlib
JOB_ID = 'stills-film-01-r7-candidates2'
TITLE = 'bicharak'
KIND = 'stills'
INPUT_ZIP_SHA256 = '1abd56dab70e580c1bfb28d612d006d937a01b6efe935b7927ce2a321a7aa8cb'
MODEL_REPO = 'black-forest-labs/FLUX.1-schnell'
MODEL_REVISION = 'resolved-at-run-time'
STEPS = 4
DRIVE_JOB_DIR = '/content/drive/MyDrive/earnalism-audiovisual/bicharak/stills-film-01-r7-candidates2'
# Hard wall-clock stop = the forecast seconds reserved on the ledger (R25)
DEADLINE_SECONDS = 1780
MANIFEST_RESERVE_S = 60
PACKAGES = ('diffusers', 'transformers', 'accelerate', 'sentencepiece', 'protobuf', 'torch')


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json, os, threading, time
BOOKKEEPING = ("progress.jsonl", "sessions.jsonl")  # resume records: never outputs


def _rows(name):
    rows = []
    path = OUT_DIR / name
    if path.is_file():
        for line in path.read_text().splitlines():
            try:
                row = json.loads(line)
            except ValueError:
                continue  # a line torn by a disconnect
            if isinstance(row, dict):
                rows.append(row)
    return rows


_APPEND_LOCK = globals().get("_APPEND_LOCK") or threading.Lock()


def _append(name, row):
    # the heartbeat thread and the main thread both append: one writer at a time
    with _APPEND_LOCK:
        path = OUT_DIR / name
        torn = path.is_file() and path.stat().st_size > 0 and path.read_bytes()[-1:] != b"\n"
        with open(path, "a") as f:
            f.write(("\n" if torn else "") + json.dumps(row, sort_keys=True) + "\n")
            f.flush()
            os.fsync(f.fileno())


def session_seconds():
    # started -> seconds, the latest line per session (this one once it has started)
    latest = {}
    for row in _rows("sessions.jsonl"):
        if "started" in row and isinstance(row.get("seconds"), (int, float)):
            latest[row["started"]] = row["seconds"]
    return latest


def note_session():
    _append("sessions.jsonl", {"started": JOB_STARTED, "seconds": round(time.time() - JOB_STARTED, 1)})


def resume_done(names):
    # Finished = present AND matching its progress.jsonl line written for THIS inputs.zip (a line
    # of other inputs proves nothing); a file without one is deleted (redone).
    logged = {}
    for row in _rows("progress.jsonl"):
        if isinstance(row.get("file"), str) and isinstance(row.get("sha256"), str) \
                and row.get("zip") == INPUT_ZIP_SHA256:
            logged[row["file"]] = row["sha256"]
    done = set()
    for name in names:
        path = OUT_DIR / name
        if not path.is_file():
            continue
        if logged.get(name) == hashlib.sha256(path.read_bytes()).hexdigest():
            done.add(name)
        else:
            path.unlink()
            print("redo", name, "(no matching progress line)")
    return done


def record_item(key, name, item_s=None):
    # item_s: the seconds this item (or its batch) took; a resumed session starts from it
    digest = hashlib.sha256((OUT_DIR / name).read_bytes()).hexdigest()
    row = {"key": key, "file": name, "sha256": digest, "zip": INPUT_ZIP_SHA256}
    if isinstance(item_s, (int, float)) and not isinstance(item_s, bool) and item_s >= 0:
        row["item_s"] = round(item_s, 1)
    _append("progress.jsonl", row)
    note_session()


def last_item_seconds():
    # The latest recorded item (or batch) time, so a resumed session's first item starts only if
    # it can finish before the stop; 0.0 before any item of this job finished.
    times = [r["item_s"] for r in _rows("progress.jsonl") if r.get("zip") == INPUT_ZIP_SHA256
             and isinstance(r.get("item_s"), (int, float))]
    return float(times[-1]) if times else 0.0


def _hard_stop():
    try:
        note_session()
    except Exception as e:  # Drive may have dropped; the runtime must still be released
        print("could not record the session seconds:", e)
    finally:
        runtime.unassign()


def _heartbeat():
    while not HEARTBEAT_STOP.wait(HEARTBEAT_S):
        try:
            note_session()  # a session killed mid-batch (OOM, disconnect) is still charged
        except Exception:
            pass


def attach_session():
    # Run once OUT_DIR is known (after the Drive mount): shorten the stop by what earlier sessions
    # used, charge this session from JOB_STARTED, and start the heartbeat.
    global PRIOR_SECONDS, EFFECTIVE_DEADLINE, STOP_GUARD, SESSION_ATTACHED
    if SESSION_ATTACHED:
        return
    PRIOR_SECONDS = sum(v for k, v in session_seconds().items() if k != JOB_STARTED)
    EFFECTIVE_DEADLINE = DEADLINE_SECONDS - PRIOR_SECONDS
    remaining = EFFECTIVE_DEADLINE - (time.time() - JOB_STARTED)
    if remaining <= 0:
        note_session()
        raise RuntimeError("reserved time used up: close the job with the CU Colab shows")
    STOP_GUARD.cancel()
    STOP_GUARD = threading.Timer(remaining, _hard_stop)
    STOP_GUARD.daemon = True
    STOP_GUARD.start()
    note_session()
    SESSION_ATTACHED = True
    threading.Thread(target=_heartbeat, daemon=True).start()
    print("reserved %d s, %d s used by earlier sessions" % (DEADLINE_SECONDS, PRIOR_SECONDS))


# R25: the hard stop is armed here, right after the config, for the full reserved seconds; the
# verify cell then calls attach_session() to take off what earlier sessions used.
HEARTBEAT_S = globals().get("HEARTBEAT_S", 20)
HEARTBEAT_STOP = globals().get("HEARTBEAT_STOP") or threading.Event()
if "JOB_STARTED" not in globals():
    JOB_STARTED = time.time()
    PRIOR_SECONDS = 0
    EFFECTIVE_DEADLINE = DEADLINE_SECONDS
    SESSION_ATTACHED = False
    from google.colab import runtime
    STOP_GUARD = threading.Timer(DEADLINE_SECONDS, _hard_stop)
    STOP_GUARD.daemon = True
    STOP_GUARD.start()
INCOMPLETE = False


def may_start(last_item_s):
    # Between items: start the next one only if it (and the manifest) fits before the hard stop.
    global INCOMPLETE
    left = EFFECTIVE_DEADLINE - (time.time() - JOB_STARTED)
    if left < last_item_s + MANIFEST_RESERVE_S:
        INCOMPLETE = True
        print("HARD STOP: %.0f s left of the %d s still reserved; no further item starts" % (left, EFFECTIVE_DEADLINE))
        return False
    return True


print("hard stop armed:", DEADLINE_SECONDS, "s from now")


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json, pathlib, zipfile
from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = pathlib.Path(DRIVE_JOB_DIR)
INPUT_ZIP = OUT_DIR / "inputs.zip"
if not INPUT_ZIP.is_file():
    raise RuntimeError("Copy inputs.zip into " + str(OUT_DIR) + " first")
actual = hashlib.sha256(INPUT_ZIP.read_bytes()).hexdigest()
if actual != INPUT_ZIP_SHA256:
    raise RuntimeError("inputs.zip sha256 mismatch: " + actual)
with zipfile.ZipFile(INPUT_ZIP) as z:
    JOB = json.loads(z.read("job.json"))
if JOB["job_id"] != JOB_ID:
    raise RuntimeError("job.json belongs to a different job")
print("inputs verified for", JOB_ID)
attach_session()


In [ ]:
# Stills: FLUX.1-schnell, 4 steps. The exact revision is resolved now and recorded.
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers", "transformers",
                "accelerate", "sentencepiece", "protobuf"], check=True)
import torch
from diffusers import FluxPipeline
from huggingface_hub import HfApi

revision = HfApi().model_info(MODEL_REPO).sha
MODEL_INFO = {"repo": MODEL_REPO, "revision": revision, "steps": STEPS}
pipe = FluxPipeline.from_pretrained(MODEL_REPO, revision=revision, torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()
EXPECTED = [item["key"] + ".png" for item in JOB["items"]]
DONE = resume_done(EXPECTED)  # finished in an earlier session: file present and sha256 logged
todo = [i for i in JOB["items"] if i["key"] + ".png" not in DONE]
print(len(DONE), "stills already done;", len(todo), "to generate")
# seconds the previous still took (from progress.jsonl in a resumed session); the next starts
# only if it fits (R25)
last = last_item_seconds()
for item in todo:
    if not may_start(last):
        break
    started = time.time()
    gen = torch.Generator("cpu").manual_seed(item["seed"])
    image = pipe(item["prompt"], num_inference_steps=STEPS, guidance_scale=0.0,
                 max_sequence_length=256, width=item["width"], height=item["height"],
                 generator=gen).images[0]
    image.save(OUT_DIR / (item["key"] + ".png"))
    last = time.time() - started
    record_item(item["key"], item["key"] + ".png", item_s=last)
    print("wrote", item["key"], "in %.1f s" % last)


In [ ]:
if RUN is not True:
    raise RuntimeError("RUN is False; set RUN = True in the first cell to run this job")
import hashlib, json, time
import importlib.metadata as _md
outputs = {}
for p in sorted(OUT_DIR.iterdir()):
    if p.is_file() and p.name not in ("manifest.json", "inputs.zip") + BOOKKEEPING:
        outputs[p.name] = hashlib.sha256(p.read_bytes()).hexdigest()
if not outputs:
    raise RuntimeError("no outputs were produced")
packages = {}  # R26: the resolved versions (and VCS commits) that actually ran
for _name in PACKAGES:
    try:
        _dist = _md.distribution(_name)
        _url = json.loads(_dist.read_text("direct_url.json") or "{}")
        _commit = _url.get("vcs_info", {}).get("commit_id")
        packages[_name] = _dist.version + (" @ " + _commit if _commit else "")
    except _md.PackageNotFoundError:
        packages[_name] = None
note_session()
_sessions = session_seconds()
complete = not INCOMPLETE and all(name in outputs for name in EXPECTED)
manifest = {"schema": "earnalism.audiovisual-outputs.v1", "job_id": JOB_ID, "title": TITLE,
            "kind": KIND, "input_zip_sha256": INPUT_ZIP_SHA256, "model": MODEL_INFO,
            "status": "COMPLETE" if complete else "INCOMPLETE",
            "deadline_seconds": DEADLINE_SECONDS,
            "elapsed_seconds": round(sum(_sessions.values()), 1), "sessions": len(_sessions),
            "packages": packages, "outputs": outputs}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")
print("manifest.json written:", manifest["status"], len(outputs), "files")
if not complete:
    print("INCOMPLETE: import refuses this folder; close the job with the CU Colab shows")
